## Chunking

Chunking 1.0
Filer över 1800 tecken chunkades med 200 overlap, medans övriga lessons behölls som naturliga semantiska chunks. Detta val gjordes för att bevara lektions strukturerna och få hög precsion i retrieval
Efter retreival testing så hittades ett problem med att korta lessons exempelvis intro lektioner kom upp med en hög semantisk träff men med tunnt innehåll. JAg gick därför vidare. (skript för detta och mer finns i Trash mappen men bör inte vara av intresse)

Chunking 2.0
För att lösa Chunking 1.0s problem så testade jag istället att mergea alla lektioner inom samma kapitel till en chunk med en max gräns på 2000 tecken. För att då bli av med dessa korta lektioner. Det jag då kom fram till var att materialets hierarki inte var konsekvent mellan kurser. I vissa kurser användes kapitel så som lektioner användes i andra kurser. Detta gjorde chunksen extremt varierande i storlek. Den största var 23213 tecken och den minsta var 146 tecken....

Efter den insikten gick jag tillbaka till att rensa data. Jag började med de minsta kapiltlen och arbetade mig upp och letade specifikt efter "intro" kapitel eller liknande. Jag lyckades då rensa bort 12 kapitel som inte tillförde någon information. Efter denna rensing blev det lättare att hitta en bra chunking logik. 

Min slutliga logik blev att 
1. Bestäm parametrar av chunks MAX_CHARS= 1800, MICRO_CHUNK_LIMIT = 500, MERGE_MAX_CHARS = 2300
2. Slå ihop alla lektioner till hela kapitel
3. Om kapitlet är mindre än MAX_CHARS spara som chunk
4. OM kapitlet är större än MAX_CHARS dela vid närmsta nya rad före MAX_CHARS tills det får plats och spara varje del som en chunk
5. Skapa DF av alla chunks
6. Loop igenom alla chunks om en chunk är mindre än MICRO_CHUNK_LIMIT slå ihop med föregående chunk men bara om den nya chunken inte blir längre än MERGE_MAX_CHARS
7. exportera

Ett problem med detta är att logiken medvetet inte splittar en stor kurs om den sakner nyrad. Detta för att jag då hällre vill granska kapitlet själv. Nu fick jag chunks som jag valde att arbeta vidare med.



In [1]:
import os
import json
from google import genai
from dotenv import load_dotenv
from google.genai import types
import numpy as np
import polars as pl

In [ ]:
def load_records_from_cleaned(folder_path):
    
    """
    Läser in alla JSON-filer från folder_path och bygger upp en lista
    av records med metadata.

    Varje record kommer att representera en fil alltså en lektion i detta fallet
    """

    records = []


    # Loopar igenom alla filer i mappen
    for root, _, files in os.walk(folder_path):
        for file in files:
            # Filtrerar på endast json filer
            if file.endswith(".json"):
                full_path = os.path.join(root, file)

                # UTF-8 så vi får med svenska tecken
                with open(full_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                    # Skapar ett strukturerat record
                    records.append({
                        "course_name": data.get("course_name"),
                        "chapter_title": data.get("chapter_title"),
                        "lesson_id": data.get("lesson_id"),
                        "lesson_title": data.get("lesson_title"),
                        "text": data.get("text"),
                        "file_path": full_path
                    })

    return records

In [ ]:
folder = r"C:\YA BI analyst\Kurser\BI25M AI & IoT\Kunskapskontroll 1\AI-IoT-Kunskapskrav-1\data\cleaned"

records = load_records_from_cleaned(folder)

# Printar ut för att se så det gått rätt till
print(f"Antal records: {len(records)}")
print(records[0])

Antal records: 349
{'course_name': 'Ansvariga- Vem ska jag kontakta?', 'chapter_title': 'Vem ska jag kontakta?', 'lesson_id': '31459054', 'lesson_title': 'Båtskada', 'text': 'Olika skeppare har olika erfarenhet av olika båtar, var därför inte rädd för att fråga föregående skeppare om saker du undrar över. Frågor om båten bör du ställa redan på bytesdagen och självklart kan du alltid höra av dig under veckan om du har ytterligare frågor. Vet du inte vem som varit på båten tidigare eller har föregående skepparen inte svar på dina frågor? Då skriver du i skeppargruppen eller en av dina mentorer.\nSkeppargruppen skall kunna användas för att fråga om eventuella strul, väder eller generella frågor. Om du mot förmodan inte får ett vettigt svar är det bäst att kontakta din\nplatsansvarig.\nHar du en skada efter till exempel en tilläggning eller liknande ska du kontakta din\nplatsansvarig samt ansvarig tekniker för din båt!', 'file_path': 'C:\\YA BI analyst\\Kurser\\BI25M AI & IoT\\Kunskapskont

In [4]:
import pandas as pd

# Ta bort tomma texter
records_clean = [r for r in records if r.get("text") and str(r.get("text")).strip()]

df = pd.DataFrame(records_clean)

# Slå ihop all text per chapter
chapter_df = (
    df.groupby(["course_name", "chapter_title"], dropna=False)
    .agg({
        "text": lambda x: "\n\n".join(x),  # slå ihop alla lessons
        "lesson_id": "count"              # antal lessons
    })
    .reset_index()
)

# Räkna storlek
chapter_df["char_count"] = chapter_df["text"].str.len()
chapter_df["word_count"] = chapter_df["text"].str.split().apply(len)

# Sortera största först
chapter_df = chapter_df.sort_values("char_count", ascending=False).reset_index(drop=True)

# 🔝 Top 20 största kapitel
print("\nStörsta kapitel:\n")
print(
    chapter_df[
        ["course_name", "chapter_title", "lesson_id", "char_count", "word_count"]
    ].head(40).to_string(index=False)
)

# 🔽 Minsta kapitel
print("\nMinsta kapitel:\n")
print(
    chapter_df.sort_values("char_count")[
        ["course_name", "chapter_title", "lesson_id", "char_count", "word_count"]
    ].head(40).to_string(index=False)
)

# 📊 Statistik
print("\nStatistik:\n")
print(f"Medel chars: {chapter_df['char_count'].mean():.0f}")
print(f"Median chars: {chapter_df['char_count'].median():.0f}")
print(f"Max chars: {chapter_df['char_count'].max()}")
print(f"Min chars: {chapter_df['char_count'].min()}")
print(f"Totalt antal kapitel: {chapter_df.shape[0]}")


Största kapitel:

                     course_name                                                         chapter_title  lesson_id  char_count  word_count
                    Båthantering                                                           Tilläggning         11       23213        4136
                    Mathantering Tidigare More Sailing recept (vissa är kvar i nya recepthäftet också)         21       20371        3595
           Destination: Kroatien                                                                   Vis          6       19853        3497
                    Mathantering                                                      Livsmedelshygien         10       19334        3004
    Genomförande av seglingsresa                                                    Säkerhetsgenomgång         13       16471        2900
                      Riktlinjer                                            Sammanfattning riktlinjer           1       16069        2446
           Dest

Efter denna check så såg jag att det fanns många kapitel som såg "värdelösa" ut. Exempelvis intro, Välkommen eller liknande. Jag började därför med att gå igenom de kapitel som såg "märkliga" ut och lista de. Här är de som jag kom fram till.

Våra båtar
Säkerhetsgenomgång med gäster
Välkommen till gänget
Introduktion riktlinjer
Utrustning på båten
Introduktion Båthantering
Nöjda Kunder
Introduktion till Segelsättning
Välkommen
Vi som jobbar på More Sailing kontoret
Tamphantering
Hur ska detta verktyg användas?


In [5]:
import pandas as pd

# Styr hur stor en chunk ska vara
MAX_CHARS = 1800

# Styr hur liten en chunk får vara
MICRO_CHUNK_LIMIT = 500

# Styr hur stor som max en chunk får bli om vi ska slå ihop micro chunks
MERGE_MAX_CHARS = 2300

# Kapitel som ska bort
# Jag har kollat så att endast 12 kapitel tas bort. För att inte råka ta bort fler kapitel med samma namn i olika kurser så hade man kunnat köra på chapter_id istället.
chapters_to_remove = [
    "Våra båtar",
    "Säkerhetsgenomgång med gäster",
    "Välkommen till gänget",
    "Introduktion riktlinjer",
    "Utrustning på båten",
    "Introduktion Båthantering",
    "Nöjda Kunder",
    "Introduktion till Segelsättning",
    "Välkommen",
    "Vi som jobbar på More Sailing kontoret",
    "Tamphantering",
    "Hur ska detta verktyg användas?"
]


def split_chapter(text, max_chars=1800):
    """
    Delar text så nära max_chars som möjligt.
    Bryter vid senaste newline (\n) före max_chars.
    Om ingen newline finns före max_chars så avbryts splitten.
    """

    text = str(text).strip()
    chunks = []

    # Delar texten om den är längre än max_chars
    while len(text) > max_chars:
        split_pos = text.rfind("\n", 0, max_chars)

        # Finns ingen radbrytning före max_chars, avbryt split
        # Vi måste alltså granska våra chunks efteråt eftersom att det finns en risk att en chunk kan bli oändligt lång.
        # Konstiga edge-case texter utan radbrytning vill vi hellre granska manuellt och leta efter en bra lösning
        if split_pos == -1:
            break

        # Lägger till chunken
        chunk = text[:split_pos].strip()
        if chunk:
            chunks.append(chunk)

        # Fortsätter
        text = text[split_pos:].strip()

    if text:
        chunks.append(text)

    return chunks


def merge_micro_chunks(chunks_df, micro_limit=500, merge_max_chars=2300):
    """
    Slår ihop små chunks med föregående chunk inom samma course_name + chapter_title.
    """
    merged_rows = []

    # Groupar på kurs och kapitel
    for (course_name, chapter_title), group in chunks_df.groupby(
        ["course_name", "chapter_title"], sort=False, dropna=False
    ):
        # Sorterar chunksen i ordning
        group = group.sort_values("chunk_index").reset_index(drop=True)

        temp_rows = []

        for _, row in group.iterrows():
            row_dict = row.to_dict()

            # Om det bara finns en chunk i kapitlet så går vi vidare
            if not temp_rows:
                temp_rows.append(row_dict)
                continue

            current_len = row_dict["char_count"]

            # Om chunken är mindre än micro_limit slå ihop det med föregående
            if current_len < micro_limit:
                merged_text = temp_rows[-1]["text"] + "\n\n" + row_dict["text"]

                # Om kontrollerar att den ihopslagna texten inte är större än merge_max_chars
                if len(merged_text) <= merge_max_chars:
                    # Om villkor uppfylts så uppdateras metadata, vilket gör det lättare för oss i retrieval test och evaluation.
                    temp_rows[-1]["text"] = merged_text
                    temp_rows[-1]["char_count"] = len(merged_text)
                    temp_rows[-1]["word_count"] = len(merged_text.split())
                else:
                    temp_rows.append(row_dict)
            else:
                temp_rows.append(row_dict)

        merged_rows.extend(temp_rows)

    return pd.DataFrame(merged_rows)


## Nu kör vi våra funktioner
# ---------------------------------------------------------------------------

# Ta bort tomma texter
records_clean = [r for r in records if r.get("text") and str(r.get("text")).strip()]

df = pd.DataFrame(records_clean)

# Filtrera bort valda kapitel
df = df[~df["chapter_title"].isin(chapters_to_remove)]

# Slår ihop all text per kapitel
# Detta valde vi att göra eftersom att chapter_id ändå inte är i ordning så kan vi inte lägga ihop lektioner semantiskt och få den strukturen
# Hade det gått så hade det varit en bättre lösning
chapter_df = (
    df.groupby(["course_name", "chapter_title"], dropna=False)
    .agg({
        "text": lambda x: "\n\n".join(x),
        # Sparar hur många lektioner som slagits ihop
        "lesson_id": "count"
    })
    .reset_index()
)

# Räkna storlek
chapter_df["char_count"] = chapter_df["text"].str.len()
chapter_df["word_count"] = chapter_df["text"].str.split().apply(len)

# Sortera största först
chapter_df = chapter_df.sort_values("char_count", ascending=False).reset_index(drop=True)

# Bygg chunks

chunks = []

# Loop igenom alla kapitel
for _, row in chapter_df.iterrows():
    course_name = row["course_name"]
    chapter_title = row["chapter_title"]
    text = row["text"]
    lesson_count = row["lesson_id"]

    # Om kapitlet är mindre än MAX_CHARS lägger vi till det
    if len(text) <= MAX_CHARS:
        chunks.append({
            "text": text,
            "course_name": course_name,
            "chapter_title": chapter_title,
            "lesson_count": lesson_count,
            "chunk_type": "full_chapter",
            "chunk_index": 1,
            "char_count": len(text)
        })
    else:
        # Om kapitlet är större än MAX_CHARS så kör vi våran split_chapter funktion
        split_chunks = split_chapter(text, max_chars=MAX_CHARS)

        # lägger in varje chunk med metadata
        for i, chunk_text in enumerate(split_chunks, start=1):
            chunks.append({
                "text": chunk_text,
                "course_name": course_name,
                "chapter_title": chapter_title,
                "lesson_count": lesson_count,
                "chunk_type": "split_chapter",
                "chunk_index": i,
                "char_count": len(chunk_text)
            })

# Skapa chunk-df
chunks_df = pd.DataFrame(chunks)
chunks_df["word_count"] = chunks_df["text"].str.split().apply(len)

# Merge micro chunks
# Försöker lägga ihop micro chunks med våran merge_micro_chunks funktion
chunks_df = merge_micro_chunks(
    chunks_df,
    micro_limit=MICRO_CHUNK_LIMIT,
    merge_max_chars=MERGE_MAX_CHARS
)

# Säkerställ word_count efter merge
chunks_df["word_count"] = chunks_df["text"].str.split().apply(len)

# Sortera största först för utskrift
chunks_df = chunks_df.sort_values("char_count", ascending=False).reset_index(drop=True)

# Chunk statistik
print("\nChunk-statistik:\n")
print(f"Totalt antal chunks: {chunks_df.shape[0]}")
print(f"Medel chars: {chunks_df['char_count'].mean():.0f}")
print(f"Median chars: {chunks_df['char_count'].median():.0f}")
print(f"Max chars: {chunks_df['char_count'].max()}")
print(f"Min chars: {chunks_df['char_count'].min()}")
print(f"Chunks < {MICRO_CHUNK_LIMIT} chars: {(chunks_df['char_count'] < MICRO_CHUNK_LIMIT).sum()}")
print(f"Chunks > {MERGE_MAX_CHARS} chars: {(chunks_df['char_count'] > MERGE_MAX_CHARS).sum()}")

print("\nStörsta chunks:\n")
print(
    chunks_df[
        ["course_name", "chapter_title", "chunk_type", "char_count", "word_count"]
    ].head(40).to_string(index=False)
)

print("\nMinsta chunks:\n")
print(
    chunks_df.sort_values("char_count")[
        ["course_name", "chapter_title", "chunk_type", "char_count", "word_count"]
    ].head(40).to_string(index=False)
)

# Bygg om input så att det passar till EmbeddingService
embedding_input = [
    {
        "text": row["text"],
        "metadata": {
            "course_name": row["course_name"],
            "chapter_title": row["chapter_title"],
            "lesson_count": int(row["lesson_count"]),
            "chunk_type": row["chunk_type"],
            "chunk_index": int(row["chunk_index"]),
            "char_count": int(row["char_count"]),
            "word_count": int(row["word_count"])
        }
    }
    for _, row in chunks_df.iterrows()
]


Chunk-statistik:

Totalt antal chunks: 250
Medel chars: 1543
Median chars: 1647
Max chars: 2203
Min chars: 587
Chunks < 500 chars: 0
Chunks > 2300 chars: 0

Största chunks:

                 course_name                                                         chapter_title    chunk_type  char_count  word_count
                  Våra båtar                                                            Dufour 530 split_chapter        2203         364
       Destination: Kroatien                                            Fastlandet Norr om Trogir  split_chapter        2112         334
                Båthantering                                               Underhåll båt, skeppare split_chapter        2087         361
       Destination: Kroatien                                        Fastlandet Trogir och söderut  split_chapter        2062         331
Genomförande av seglingsresa                                                    Säkerhetsgenomgång split_chapter        1944         342
   

## Embeddings
Här testade jag först med gemini embedding men blev snabbt problem med request limits. Detta löste jag igenom att sätta köra det i mindre batches. Då kom jag under gratis versionens gränser och slapp köra det på en lokal modell.

### EmbeddingService och VectorStore
Jag valde också att implementera EmbeddingService klass för att skapa en mer strukturerad och återanvändbar lösning.

Detta gav flera fördelar:

Inkapslad logik – all hantering av embeddings (API-anrop, batching, rate limiting och export) finns samlad på ett ställe
Återanvändbarhet – klassen kan användas i flera notebooks utan att duplicera kod
Konfigurerbarhet – modell, parametrar och beteende kan enkelt ändras centralt
Läsbarhet – tydligare struktur jämfört med spridda funktioner

Hur klassen är uppbyggd kan du se i embedding_service filen.

### VectorStore
Jag valde att göra en liknande lösning med VectorStore som en klass för att hantera lagring och sökning av embeddings på ett strukturerat och effektivt sätt.

Detta gav följande fördelar:

Inkapslad logik – all hantering av embeddings (inläsning, konvertering och sökning) finns samlad på ett ställe
Återanvändbarhet – klassen kan enkelt användas i olika delar av projektet utan att duplicera kod
Tydlig separation av ansvar – denna klass ansvarar enbart för retrieval-delen i RAG-pipelinen

Hur klassen är uppbyggd kan du se i vector_store filen.

In [ ]:
#Importer och variabler
from embedding_service import EmbeddingService
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

embedding_service = EmbeddingService(client, types)

In [ ]:
# Anroppar embedding_service för att embedda våra chunks med delay
embedded_docs = embedding_service.embed_documents_with_delay(
    embedding_input,
    batch_size=50,
    wait_seconds=65
)

Embedding 1/250
Embedding 2/250
Embedding 3/250
Embedding 4/250
Embedding 5/250
Embedding 6/250
Embedding 7/250
Embedding 8/250
Embedding 9/250
Embedding 10/250
Embedding 11/250
Embedding 12/250
Embedding 13/250
Embedding 14/250
Embedding 15/250
Embedding 16/250
Embedding 17/250
Embedding 18/250
Embedding 19/250
Embedding 20/250
Embedding 21/250
Embedding 22/250
Embedding 23/250
Embedding 24/250
Embedding 25/250
Embedding 26/250
Embedding 27/250
Embedding 28/250
Embedding 29/250
Embedding 30/250
Embedding 31/250
Embedding 32/250
Embedding 33/250
Embedding 34/250
Embedding 35/250
Embedding 36/250
Embedding 37/250
Embedding 38/250
Embedding 39/250
Embedding 40/250
Embedding 41/250
Embedding 42/250
Embedding 43/250
Embedding 44/250
Embedding 45/250
Embedding 46/250
Embedding 47/250
Embedding 48/250
Embedding 49/250
Embedding 50/250

Batch klar. Väntar 65 sekunder...

Embedding 51/250
Embedding 52/250
Embedding 53/250
Embedding 54/250
Embedding 55/250
Embedding 56/250
Embedding 57/250
Embe

In [ ]:
#Använder embedding_service olika funktioner för att spara ner våra embeddings
df = embedding_service.to_polars_df(embedded_docs)
print(df.head())
embedding_service.save_to_parquet(embedded_docs, "embeddings_v2.parquet")

shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ text      ┆ embedding ┆ course_na ┆ chapter_t ┆ … ┆ chunk_typ ┆ chunk_ind ┆ char_coun ┆ word_cou │
│ ---       ┆ ---       ┆ me        ┆ itle      ┆   ┆ e         ┆ ex        ┆ t         ┆ nt       │
│ str       ┆ list[f64] ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ i64       ┆ i64       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Våra      ┆ [-0.00048 ┆ Våra      ┆ Dufour    ┆ … ┆ split_cha ┆ 1         ┆ 2203      ┆ 364      │
│ Dufour    ┆ 5,        ┆ båtar     ┆ 530       ┆   ┆ pter      ┆           ┆           ┆          │
│ 530 är    ┆ -0.03398, ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ väldigt   ┆ … -0.009… ┆           ┆           ┆   ┆           ┆           ┆